# Exponential `CURVE FIT` function algorithm for `pole_to_pole distance for mean fit data` 
### This script fits an exponential curve to the mean of each cell type of the *C. elegans* embryo in batch process. This curve uses the sigmoid fitting function where *`y = L / (1 + np.exp(-k*(x-x0))) + b`*. The goal for fitting a mathematical function to the mean value is to determine the *Initial pole-to-pole length*, *Final pole-to-pole length* and *Elongation rate* at the required time point. 

#### `INPUT FILES` The csv files of each cell type containing the pole-to-pole distance data of different experiments (n-value). 
#### `OUTPUT FILES` *FIRST_GROUP_OUTPUT_FILES*: The *.png* files of the fitted plot of each cell type. *SECOND_GROUP_OUTPUT_FILE*: A *.csv* file containing the **Initial pole_to_pole length (µm)**, the **Final pole_to_pole length (µm)** and the **Elongation rate (µm/minute)** of each of the cell type. 

In [1]:
# library packages
import os
import warnings
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from numpy import exp, linspace, random, arange
from scipy.optimize import curve_fit, least_squares

In [2]:
# input folder
folder = r'D:\data\Analysis Data\python_analysis\input'
fileTable = os.listdir(folder)

# output folder
save_files = r'D:\data\Analysis Data\python_analysis\output'

In [3]:
# create a new DataFrame to append the new generated table 
fit_Result = pd.DataFrame()

# read out individual files and compute for different operations as defined in the for loop
for file in os.scandir(folder):
    df = pd.read_csv(file)
    
    # create a new dataframe 
    Exp_Column = df.loc[:, df.columns.str.startswith('Exp')]
    mean_column = df.loc[:, df.columns.str.startswith('mean')]
    time_column = df.loc[:, df.columns.str.startswith('time')]
    
    df_new = [Exp_Column, mean_column, time_column]
    df_Table = pd.concat(df_new, axis=1)
    
    '''
    drop all the rows were n-value is less than 3 the mean and the time 
    columns are included, ie, the number of rows to be computed should be >= 5
    '''
    newTable = df_Table.dropna(thresh=3) 
    # print(newTable)
    
    # define a logistics function to fit for a sigmoid curved data
    
    '''
    x = independent variable [time or position]
    L = maximum value that the dependent variable can take (also known as the saturation level or the upper asymptote)
    x0 = x-value of the sigmoid's midpoint (the inflection point)
    k = rate of growth (controlling the steepness of the curve around the inflection point)
    b = constant (determines the lower asymptote or the minimum value of the curve)
    '''
    def sigmoid(x, L ,x0, k, b):
        y = L / (1 + np.exp(-k*(x-x0))) + b
        return y
    
    # ignore warning
    # warnings.simplefilter(action="ignore", category=FutureWarning)
    warnings.filterwarnings("ignore") 

    # loop through the desired columns header
    cells= [columnname for columnname in newTable if columnname.startswith('Exp')]
    # print(cells)

    # define x-values and y-values 
    x = newTable['time']
    y_mean = newTable['mean']
    
    # compute the initial guess
    initial_guess = [max(y_mean)-min(y_mean), 5, 0.05, min(y_mean)]

    # summarize the parameter
    popt, pcov = curve_fit(sigmoid, x, y_mean, initial_guess, maxfev=10000)
    
    ''' 
    Define a sequence of inputs between the smallest and largest known inputs and define the fit. 
    Let the maximum input assume the maximum values of x. 
    '''
    x_fit = np.arange(x.min()-0.01, x.max()+0.01, 0.01)
    y_fit = sigmoid(x_fit, *popt)
    
    plt.figure(figsize=(5,5))
    
    # plot the primary data
    for i_columns in cells: 
        ax1 = sns.scatterplot(data=newTable, x=x, y=i_columns, color='grey', alpha=0.2)
        
    # add the desired features on the plot
    ax1.set_xlabel('time [s]', fontsize= 18)
    ax1.set_ylabel('distance [μm]', fontsize= 18)
    plot_title = (file.name).split('.')[0]
    ax1.axes.set_title(plot_title, fontsize= 20, fontweight='bold')  
    ax1.set(ylim=(0, 30), xlim=(-120, 220))
    
    # plot the mean on the primary plot
    sns.scatterplot(data=newTable, x=x, y=y_mean, color='cyan', alpha=0.6, ax=ax1)
    
    # plot the fit on the primary plot
    sns.lineplot(x_fit, y_fit, alpha = 1, color='magenta', ax=ax1)
    
    # add a vertical line at time 0 sec to indicate anaphase onset
    ax1.vlines(x=0, ymin=0, ymax=30, linestyle='solid', color='red')
    
    # plot_files = os.path.join(save_files)
    plotfile = (file.name).split('.')[0] + '.png'
    plt.savefig(os.path.join(save_files, plotfile), dpi=300)
    
    # assign variables to the fit parameter output
    L = popt[0]
    x0 = popt[1]
    k = popt[2]
    b = popt[3]
    
    initial_length = b
    final_length = b + L
    elongation_rate = ((L*k)/4)*60 # multiple by 60 to give the final value in µm/minute
    
    # create a new dictionary for each parmeter and add to the dataframe fit_Result
    parameters = {'Initial pole_pole length (µm)': initial_length, 
                  'Final pole_pole length (µm)': final_length, 
                  'Elongation rate (µm/min)': elongation_rate}
    parameters_df = pd.DataFrame.from_dict(parameters, orient='index', columns=[file.name.split('.')[0]])
    fit_Result = pd.concat([fit_Result, parameters_df], axis=1)
    
# transpose the table
fit_Result_transpose = (fit_Result).T

# assign header to the index column
fit_Result_transpose.index.names = ['Cells']

# save the table, fit_Result, to a csv file
fit_Result_transpose.to_csv(os.path.join(save_files, 'Fit_Result_pole_pole.csv'), encoding='utf-8')

plt.close('all')